# AMBA -- 01. Ambil Data dari Supabase

Mengunduh enam tabel mentah dari Supabase (baca saja) ke cache lokal
`data/raw/`. Notebook yang sama dipakai untuk pengambilan pertama kali
maupun pembaruan harian -- bedanya hanya seberapa banyak baris yang
sudah tersedia di Supabase saat ini dijalankan.

Tidak ada operasi tulis ke Supabase di sini atau di jalur ini --
seluruhnya select(). Ini jalur BACA yang independen dari
`sectors_fetcher/main.py` (jalur TULIS, butuh `sectors_fetcher/storage/`
yang belum ada di checkout ini -- lihat README bagian Keterbatasan).
Lihat `sectors_fetcher/supabase_io.py`.


In [ ]:
import sys
from pathlib import Path


def _cari_root(mulai: Path) -> Path:
    for kandidat in [mulai, *mulai.parents]:
        if (kandidat / "sectors_fetcher" / "__init__.py").exists():
            return kandidat
    raise RuntimeError(
        "Tidak menemukan folder 'sectors_fetcher/' di direktori ini atau induknya. "
        "Jalankan notebook dari dalam folder proyek."
    )


ROOT = _cari_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd

from sectors_fetcher import config, supabase_io

print(f"Root proyek: {ROOT}")

## 1. Periksa kredensial Supabase

In [ ]:
if not config.kredensial_lengkap():
    raise RuntimeError(
        "SUPABASE_URL dan/atau SUPABASE_KEY belum terisi.\n\n"
        "Isi lewat salah satu cara berikut, lalu jalankan ulang cell ini:\n"
        "  1. Buat file .env di folder sectors_fetcher/, isi kedua nilainya.\n"
        "  2. Atau ekspor langsung di shell sebelum menjalankan Jupyter:\n"
        "     export SUPABASE_URL='https://xxxx.supabase.co'\n"
        "     export SUPABASE_KEY='xxxx'\n\n"
        "Lihat README bagian 'Menjalankan' untuk detail."
    )

client = supabase_io.get_client()
assert client is not None
print("Kredensial ditemukan, klien Supabase siap.")

## 2. Unduh enam tabel mentah (baca saja)

In [ ]:
tabel_yang_diunduh = {nama_tabel: None for nama_tabel in config.SUPABASE_TABLES.values()}

for nama_tabel in tabel_yang_diunduh:
    df = supabase_io.download_table(client, nama_tabel)
    tabel_yang_diunduh[nama_tabel] = df
    print(f"{nama_tabel:32s} {len(df):>8,} baris")

## 3. Sesuaikan nama kolom tabel suspensi (kalau berbeda) & validasi skema

Seluruh kode di proyek memakai nama kolom `event_date` dan `reason` untuk
tabel suspensi. `sectors_fetcher.config.KOLOM_TANGGAL_SUSPENSI` sudah
diisi `"suspension_date"` -- kalau tabel Anda beda, ganti nilai itu
(dan/atau `KOLOM_ALASAN_SUSPENSI`) sekali di `sectors_fetcher/config.py`.
Kelima tabel lain divalidasi lewat `supabase_io.py::validasi_kolom_tabel`.


In [ ]:
nama_tabel_suspensi = config.SUPABASE_TABLES["suspensions"]
df_suspensi = supabase_io.normalisasi_tabel_suspensi(tabel_yang_diunduh[nama_tabel_suspensi])
tabel_yang_diunduh[nama_tabel_suspensi] = df_suspensi
print(f"Kolom {nama_tabel_suspensi}: {list(df_suspensi.columns)}")

for nama_tabel, df in tabel_yang_diunduh.items():
    if nama_tabel == nama_tabel_suspensi:
        continue
    supabase_io.validasi_kolom_tabel(df, nama_tabel)
print("Skema kelima tabel lain sesuai yang dibutuhkan.")

## 4. Simpan cache lokal (CSV)

In [ ]:
config.RAW_DIR.mkdir(parents=True, exist_ok=True)
for nama_tabel, df in tabel_yang_diunduh.items():
    df.to_csv(config.RAW_DIR / f"{nama_tabel}.csv", index=False)

print(f"Enam tabel disimpan ke {config.RAW_DIR}")
print("Lanjut ke eda_and_feature_engineering.ipynb untuk feature engineering.")